# Definição de qual subconjunto cada município de teste pertence

Para cada algoritmo de clustering (KMeans, Hierárquico, DBSCAN), calculamos o centróide de cada cluster a partir do **treino** e atribuímos cada município do **teste** ao cluster do centróide mais próximo (distância euclidiana no espaço padronizado). Cada município de teste passa a fazer parte do conjunto de teste daquele subconjunto específico.

In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

base_path = Path.cwd().parent
dir_processed = base_path / "data" / "processed"

df_teste = pd.read_csv(dir_processed / "Base_de_Dados_Teste.csv", index_col="id_municipio")

with open(dir_processed / "features_perfil.json", encoding="utf-8") as f:
    features_perfil = json.load(f)

print(f"Municípios de teste: {len(df_teste)}")
print(f"Variáveis de perfil: {len(features_perfil)}")

Municípios de teste: 1114
Variáveis de perfil: 24


### Função de atribuição por centróide mais próximo

O centróide é calculado só com os clusters "reais" de cada algoritmo — o cluster `-1` (Atípico/Ruído, presente no Hierárquico e no DBSCAN) fica de fora do cálculo de centróide de propósito: não existe um "centróide do ruído" com sentido geométrico, e forçar um município de teste nesse grupo por proximidade de centróide contradiria a própria definição de atípico/ruído. Isso tem uma consequência importante, detalhada mais abaixo.

In [5]:
def atribuir_por_centroide(X_treino_scaled, labels_treino, X_teste_scaled):
    """
    Calcula o centróide de cada cluster a partir do treino e atribui
    cada registro de teste ao cluster do centróide mais próximo
    (distância euclidiana no espaço padronizado).
    """
    clusters_validos = sorted(c for c in np.unique(labels_treino) if c != -1)

    centroides = np.array([
        X_treino_scaled[labels_treino == c].mean(axis=0)
        for c in clusters_validos
    ])

    # distância de cada ponto de teste até cada centróide
    dists = np.linalg.norm(
        X_teste_scaled[:, None, :] - centroides[None, :, :], axis=2
    )
    idx_mais_proximo = dists.argmin(axis=1)

    return np.array([clusters_validos[i] for i in idx_mais_proximo])

### Aplicação para os três algoritmos

Para cada algoritmo: carregamos `treino_rotulado.csv` (gerado no respectivo notebook 05), reajustamos o **mesmo** `SimpleImputer`/`StandardScaler` (mediana + padronização) só com o treino — nunca com o teste, para não vazar informação — e aplicamos a função acima. Como o treino carregado é exatamente o mesmo CSV usado no notebook 05 (que por sua vez veio de `Base_de_Dados_Treino.csv`), reajustar o imputer/scaler aqui reproduz fielmente os valores usados no clustering original.

In [6]:
ALGORITMOS = {
    'kmeans':      'Cluster_Perfil',
    'hierarquico': 'Cluster_Perfil_Hierarquico',
    'dbscan':      'Cluster_Perfil_DBSCAN',
}

resultados_por_algoritmo = {}

for nome_algo, coluna_cluster in ALGORITMOS.items():
    dir_algo = dir_processed / "subconjuntos" / nome_algo
    df_treino_rotulado = pd.read_csv(dir_algo / "treino_rotulado.csv", index_col="id_municipio")

    imputer = SimpleImputer(strategy='median')
    X_treino_imputed = imputer.fit_transform(df_treino_rotulado[features_perfil])

    scaler = StandardScaler()
    X_treino_scaled = scaler.fit_transform(X_treino_imputed)

    labels_treino = df_treino_rotulado[coluna_cluster].values

    X_teste_imputed = imputer.transform(df_teste[features_perfil])
    X_teste_scaled = scaler.transform(X_teste_imputed)

    labels_teste = atribuir_por_centroide(X_treino_scaled, labels_treino, X_teste_scaled)

    resultados_por_algoritmo[nome_algo] = {
        'coluna_cluster': coluna_cluster,
        'df_treino_rotulado': df_treino_rotulado,
        'labels_treino': labels_treino,
        'labels_teste': labels_teste,
    }

    print(f"{nome_algo}: {len(np.unique(labels_treino[labels_treino != -1]))} clusters válidos no treino, "
          f"teste atribuído a {len(np.unique(labels_teste))} clusters distintos")

kmeans: 6 clusters válidos no treino, teste atribuído a 6 clusters distintos
hierarquico: 4 clusters válidos no treino, teste atribuído a 4 clusters distintos
dbscan: 2 clusters válidos no treino, teste atribuído a 2 clusters distintos


### Exportação dos subconjuntos de teste, por algoritmo e por cluster

Para **todo** cluster encontrado no treino (inclusive os que, por azar da atribuição, não recebam nenhum município de teste) é gerado um arquivo — mesmo que vazio — para deixar explícito, e não silenciosamente ausente, quando um subconjunto não tem teste correspondente. Os arquivos de teste têm **todas as features**, no mesmo formato dos subconjuntos de treino gerados no notebook 05.

In [7]:
resumo_geral = []

for nome_algo, resultado in resultados_por_algoritmo.items():
    dir_algo = dir_processed / "subconjuntos" / nome_algo
    coluna_cluster = resultado['coluna_cluster']
    labels_treino = resultado['labels_treino']
    labels_teste = resultado['labels_teste']

    df_teste_rotulado = df_teste.copy()
    df_teste_rotulado[coluna_cluster] = labels_teste
    df_teste_rotulado.to_csv(dir_algo / "teste_rotulado.csv", index_label="id_municipio")

    # todos os clusters REAIS do treino (o -1, quando existe, nunca recebe teste -- ver nota acima)
    clusters_do_treino = sorted(c for c in np.unique(labels_treino) if c != -1)

    for cluster_id in clusters_do_treino:
        n_treino = int((labels_treino == cluster_id).sum())
        subconjunto_teste = df_teste_rotulado[df_teste_rotulado[coluna_cluster] == cluster_id]
        n_teste = len(subconjunto_teste)

        nome_arquivo = f"teste_cluster_{cluster_id}.csv"
        subconjunto_teste.to_csv(dir_algo / nome_arquivo, index_label="id_municipio")

        resumo_geral.append({
            'algoritmo': nome_algo,
            'cluster': cluster_id,
            'n_treino': n_treino,
            'n_teste': n_teste,
        })

        aviso = "  <-- SEM municípios de teste!" if n_teste == 0 else ""
        print(f"[{nome_algo}] cluster {cluster_id}: treino={n_treino}, teste={n_teste}{aviso}")

    # registra o -1 (quando existe) só para deixar visível que ele NÃO tem teste, por decisão de projeto
    if -1 in np.unique(labels_treino):
        n_treino_atipico = int((labels_treino == -1).sum())
        resumo_geral.append({
            'algoritmo': nome_algo,
            'cluster': -1,
            'n_treino': n_treino_atipico,
            'n_teste': 0,
        })
        print(f"[{nome_algo}] cluster -1 (Atípico/Ruído): treino={n_treino_atipico}, "
              f"teste=0 (por definição -- não entra na atribuição por centróide)")

df_resumo_geral = pd.DataFrame(resumo_geral)
display(df_resumo_geral)

[kmeans] cluster 0: treino=1188, teste=283
[kmeans] cluster 1: treino=831, teste=224
[kmeans] cluster 2: treino=606, teste=139
[kmeans] cluster 3: treino=302, teste=72
[kmeans] cluster 4: treino=1289, teste=324
[kmeans] cluster 5: treino=240, teste=72
[hierarquico] cluster 0: treino=647, teste=177
[hierarquico] cluster 1: treino=568, teste=135
[hierarquico] cluster 2: treino=2919, teste=727
[hierarquico] cluster 3: treino=215, teste=75
[hierarquico] cluster -1 (Atípico/Ruído): treino=107, teste=0 (por definição -- não entra na atribuição por centróide)
[dbscan] cluster 0: treino=3504, teste=1092
[dbscan] cluster 1: treino=10, teste=22
[dbscan] cluster -1 (Atípico/Ruído): treino=942, teste=0 (por definição -- não entra na atribuição por centróide)


,algoritmo,cluster,n_treino,n_teste
0,kmeans,0,1188,283
1,kmeans,1,831,224
2,kmeans,2,606,139
3,kmeans,3,302,72
4,kmeans,4,1289,324
5,kmeans,5,240,72
6,hierarquico,0,647,177
7,hierarquico,1,568,135
8,hierarquico,2,2919,727
9,hierarquico,3,215,75


### O que fazer com o cluster `-1` (Atípico/Ruído)

Ele **tem** subconjunto de treino (exportado no notebook 05), mas fica **sem** subconjunto de teste com o método de centróide mais próximo — não é um bug, é consequência direta da definição de centróide não se aplicar a ruído. Isso é uma decisão de projeto que vale levar ao seu orientador antes de seguir para a modelagem, com três caminhos possíveis:

1. **Descartar** o subconjunto `-1` da avaliação (não treinar/testar modelo para atípicos).
2. **Avaliar com o modelo geral** (treinado na base toda) os municípios de teste mais distantes de todos os centróides — ou seja, redefinir "atípico no teste" por um limiar de distância, em vez de deixar simplesmente sem teste.
3. **Incluir `-1` no cálculo do centróide** mesmo assim (tratar como só mais um grupo) — tecnicamente simples (bastaria remover o filtro `if c != -1` da função), mas semanticamente questionável, já que o "centróide do ruído" não representa uma região densa e coesa como os demais.

Nenhuma das três está implementada por padrão aqui — a função ficou fiel ao que foi pedido (nearest centroid, ignorando ruído), e essa é a lacuna que falta fechar com uma decisão explícita.